# All studied countries extended migration analysis

This notebook applies the same extended workflow to every studied country folder: `FRA`, `GRC`, `TUR`, `ITA`, and `GBR`.

For every country, it builds migration mentions, concreteness scores, policy diffusion edges, policy agency mechanisms, narrative frames, internal/external direction labels, high-concreteness event tables, and interactive visualizations. Outputs are written to separate country directories so results do not overwrite each other.

In [ ]:
import sys
from pathlib import Path

# Explanation: Make src and scripts importable whether Jupyter starts in the notebook folder or project root.
NOTEBOOK_ROOT = Path.cwd()
PROJECT_ROOT = NOTEBOOK_ROOT.parent if NOTEBOOK_ROOT.name == "notebooks" else NOTEBOOK_ROOT
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import polars as pl
from IPython.display import HTML, Image, Markdown, display

import run_extended_analysis as runner
from src import data_model, visualize
from src.config import PROCESSED_DIR

COUNTRIES = runner.DEFAULT_COUNTRIES

## Method notes

The notebook reuses `scripts/run_extended_analysis.py`, which is the script version of the France extended notebook. It discovers available `Table1_Fact/<COUNTRY>/<COUNTRY>_<YEAR>_facts.parquet` files, then applies the same analytical modules:

- migration relevance filtering from topic labels and migration keywords
- country/entity metadata and domestic self-mention removal
- concreteness scoring and high-concreteness event extraction
- migrant cohort and policy measure classification
- policy agency classification: learning/FROM, coercion/TO, competition, cooperation/exchange, neutral reporting
- narrative framing and argument-scheme classification
- internal/inbound vs external/transnational migration direction
- country-specific tables, maps, timelines, networks, and HTML dashboards

In [ ]:
# Explanation: Inspect available country folders and years before running the full pipeline.
country_years = {}
for country in COUNTRIES:
    files = runner.available_fact_files(country)
    years = [runner.year_from_fact_path(path) for path in files]
    country_years[country] = years
    print(f"{country}: {min(years)}-{max(years)} | {len(files)} files")
    for path in files:
        print(f"  - {path.name}")

In [ ]:
# Explanation: Run the same extended pipeline for every added country folder.
# Set RUN_PIPELINE = False when you only want to inspect already generated outputs.
RUN_PIPELINE = True

country_outputs = {}
if RUN_PIPELINE:
    for country, years in country_years.items():
        print(f"\n=== {country} {min(years)}-{max(years)} ===")
        country_outputs[country] = runner.save_country_analysis(country, years)
        for name, path in country_outputs[country].items():
            print(f"{name}: {path}")

In [ ]:
# Explanation: Summarize processed mention counts and key output availability.
summary_rows = []
for country, years in country_years.items():
    prefix = runner.country_prefix(country, years)
    country_dir = PROCESSED_DIR / prefix
    mentions_path = country_dir / f"{prefix}_migration_mentions_extended.parquet"
    if not mentions_path.exists():
        summary_rows.append({"country": country, "status": "missing", "mentions": 0})
        continue
    mentions = pl.read_parquet(mentions_path)
    summary_rows.append({
        "country": country,
        "years": f"{min(years)}-{max(years)}",
        "mentions": mentions.height,
        "mean_concreteness": mentions.get_column("concreteness_score").mean(),
        "policy_agency_edges": (country_dir / f"{prefix}_policy_agency_edges.csv").exists(),
        "high_concreteness_events": (country_dir / f"{prefix}_high_concreteness_events.csv").exists(),
        "advanced_figures": (country_dir / "figures_interactive_advanced").exists(),
    })

summary = pl.DataFrame(summary_rows)
with pl.Config(tbl_rows=20, tbl_cols=20):
    display(summary)

## Country reference profiles

These tables show which other countries/entities each parliament mentions most often, with concreteness and discourse labels attached. This is the direct analogue of the France country-profile table.

In [ ]:
profile_tables = []
for country, years in country_years.items():
    prefix = runner.country_prefix(country, years)
    path = PROCESSED_DIR / prefix / f"{prefix}_country_mention_profile.csv"
    if path.exists():
        profile_tables.append(
            pl.read_csv(path)
            .with_columns(pl.lit(country).alias("source_country"))
            .select(pl.col("source_country"), pl.all().exclude("source_country"))
            .head(20)
        )

country_profiles = pl.concat(profile_tables, how="diagonal") if profile_tables else pl.DataFrame()
with pl.Config(tbl_rows=100, tbl_cols=24, fmt_str_lengths=80):
    display(country_profiles)

## High-concreteness facts

These rows are the most concrete migration-related snippets: real places, dates, organizations, named events, or bounded incidents. They are the source table for the fact timeline and visibility maps.

In [ ]:
fact_tables = []
wanted_cols = [
    "source_country", "session_date", "entity_content", "event_label",
    "proper_noun_anchors", "countries_detected_in_context", "narrative_frame",
    "policy_agency_type", "migration_direction", "fact_concreteness_score",
    "context_window",
]

for country, years in country_years.items():
    prefix = runner.country_prefix(country, years)
    path = PROCESSED_DIR / prefix / f"{prefix}_high_concreteness_events.csv"
    if path.exists():
        facts = pl.read_csv(path).with_columns(pl.lit(country).alias("source_country"))
        present_cols = [col for col in wanted_cols if col in facts.columns]
        fact_tables.append(facts.select(present_cols).head(25))

high_concreteness_facts = pl.concat(fact_tables, how="diagonal") if fact_tables else pl.DataFrame()
with pl.Config(tbl_rows=120, tbl_cols=20, fmt_str_lengths=180):
    display(high_concreteness_facts)

## Policy agency summaries

This section shows how each source parliament positions other countries/entities: as models to learn from, targets of pressure, cooperation partners, competitors, or neutral reported cases.

In [ ]:
agency_tables = []
for country, years in country_years.items():
    prefix = runner.country_prefix(country, years)
    path = PROCESSED_DIR / prefix / f"{prefix}_policy_agency_edges.csv"
    if path.exists():
        agency_edges = pl.read_csv(path)
        summary = (
            agency_edges
            .group_by(["source_country", "target_entity", "policy_agency_type"])
            .agg(pl.col("weight").sum().alias("mentions"))
            .sort(["source_country", "mentions"], descending=[False, True])
            .head(40)
        )
        agency_tables.append(summary)

policy_agency_summary = pl.concat(agency_tables, how="diagonal") if agency_tables else pl.DataFrame()
with pl.Config(tbl_rows=180, tbl_cols=12, fmt_str_lengths=80):
    display(policy_agency_summary)

## Dyadic data model: resolver, bilateral matrix, asymmetry, shocks

This section fixes the source-country to target-country model before any further interpretation. It resolves `entity_content` to `target_country_iso3`, filters at `speech_id` level for migration speeches, computes mention-window concreteness, and then builds the bilateral matrix and reciprocal asymmetry table.

In [ ]:
# Explanation: Build the resolver once. It is a regular parquet table used by lazy joins.
resolver_path = data_model.save_country_resolver()
resolver = data_model.scan_country_resolver(resolver_path)
resolver_preview = resolver.collect()
print(f"Resolver rows: {resolver_preview.height:,}")
with pl.Config(tbl_rows=30, tbl_cols=8, fmt_str_lengths=80):
    display(resolver_preview.head(30))

In [ ]:
# Explanation: Combine the added country facts lazily, resolve target countries, and keep migration speeches.
all_fact_frames = [runner.load_country_facts(country, years) for country, years in country_years.items()]
all_facts = pl.concat(all_fact_frames, how="vertical")

facts_mig = data_model.migration_country_mentions(
    all_facts,
    country_resolver=resolver,
    min_hits=2,
    min_confidence=0.85,
)
mentions_concrete = data_model.mentions_with_concreteness(facts_mig, all_facts)

dyadic_dir = PROCESSED_DIR / "ALL_AVAILABLE_COUNTRIES_dyadic_data_model"
dyadic_dir.mkdir(parents=True, exist_ok=True)
mentions_path = dyadic_dir / "resolved_migration_country_mentions.parquet"
mentions_concrete.sink_parquet(mentions_path)
print(f"Saved resolved mention-level data to {mentions_path}")

In [ ]:
# Explanation: Build the basic bilateral matrix and the concreteness matrix from the resolved mention table.
resolved_mentions = pl.scan_parquet(mentions_path)
bilateral = data_model.bilateral_matrix(resolved_mentions)
bilateral_concrete = data_model.bilateral_concreteness(resolved_mentions)
asymmetry = data_model.compute_asymmetry(bilateral, min_total_traffic=20)

bilateral.write_csv(dyadic_dir / "bilateral_matrix.csv")
bilateral_concrete.write_csv(dyadic_dir / "bilateral_concreteness.csv")
asymmetry.write_csv(dyadic_dir / "asymmetry_table.csv")

with pl.Config(tbl_rows=80, tbl_cols=16, fmt_str_lengths=80):
    display(bilateral.head(80))
    display(bilateral_concrete.head(80))
    display(asymmetry.head(80))

In [ ]:
# Explanation: Shock windows show which source-target arcs switch on before/after major migration-related events.
shock_tables = {}
for shock_name in ["moria_fire", "belarus_crisis", "ukraine_war"]:
    shock_tables[shock_name] = data_model.shock_window_matrix(
        resolved_mentions,
        data_model.SHOCKS[shock_name],
        window_days=90,
    )
    shock_tables[shock_name].write_csv(dyadic_dir / f"shock_{shock_name}_window.csv")
    display(Markdown(f"### {shock_name}"))
    with pl.Config(tbl_rows=40, tbl_cols=14, fmt_str_lengths=80):
        display(shock_tables[shock_name].head(40))

In [ ]:
# Explanation: Save and display the new dyadic-model illustrations.
data_model_figures = visualize.save_data_model_figures(
    bilateral_concrete,
    asymmetry,
    dyadic_dir,
    shock_tables=shock_tables,
)
for name, path in data_model_figures.items():
    print(f"{name}: {path}")
    if str(path).endswith(".png"):
        display(Image(filename=str(path)))

## Cross-country comparison visualizations

These figures compare all available studied parliaments directly: volume, shared targets, concreteness, agency, narrative polarity, direction, cohorts, policy measures, sentiment, and concrete fact evidence.

In [ ]:
# Explanation: Display all cross-country comparison figures generated by scripts/generate_all_country_visualizations.py.
comparison_dir = PROCESSED_DIR / "ALL_AVAILABLE_COUNTRIES_comparisons"
comparison_summary = comparison_dir / "comparison_summary_table.html"
if comparison_summary.exists():
    display(HTML(f'<a href="{comparison_summary.resolve()}">Cross-country comparison summary table</a>'))

fig_dir = comparison_dir / "figures_cross_country"
for html_path in sorted(fig_dir.rglob("*.html")):
    display(HTML(f'<a href="{html_path.resolve()}">{html_path.name}</a>'))
for png_path in sorted(fig_dir.glob("*.png")):
    print(png_path)
    display(Image(filename=str(png_path)))

## Interactive visualizations

This mirrors the France notebook display logic, but loops through each added country. It prints every interactive HTML file and displays every PNG from both the standard extended figures and the advanced interactive figure folder.

In [ ]:
for country, years in country_years.items():
    prefix = runner.country_prefix(country, years)
    display(Markdown(f"### {country} {min(years)}-{max(years)}"))
    html_links = []
    for folder in ["figures_altair_extended", "figures_interactive_advanced"]:
        fig_dir = PROCESSED_DIR / prefix / folder
        for path in sorted(fig_dir.glob("*.html")):
            html_links.append(f'<li><a href="{path.resolve()}">{folder}/{path.name}</a></li>')
    if html_links:
        display(HTML(f"<ul>{''.join(html_links)}</ul>"))
    else:
        display(Markdown("No interactive figures found yet. Run the pipeline cell above."))

In [ ]:
# Explanation: Display all saved PNG illustrations inline, exactly as the France notebook does.
for country, years in country_years.items():
    prefix = runner.country_prefix(country, years)
    display(Markdown(f"### {country} saved PNG illustrations"))
    for folder in ["figures_altair_extended", "figures_interactive_advanced"]:
        fig_dir = PROCESSED_DIR / prefix / folder
        png_paths = sorted(fig_dir.glob("*.png"))
        if png_paths:
            display(Markdown(f"#### {folder}"))
        for path in png_paths:
            print(path)
            display(Image(filename=str(path)))

## One-page visualization index

The script below regenerates the complete all-country visualization bundle and writes one HTML index that links every per-country and combined dyadic figure.

In [ ]:
# Explanation: Open this index to browse all visualizations generated by the code.
visualization_index = PROCESSED_DIR / "all_studied_countries_visualization_index.html"
display(HTML(f'<a href="{visualization_index.resolve()}">All studied countries visualization index</a>'))
print(visualization_index)